<a href="https://colab.research.google.com/github/zencolab/WhatDreamsCost-ComfyUI/blob/main/MiniMax_H3_ComfyUI_Colab_A100_v3_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MiniMax-H3 · ComfyUI on Colab **A100** · v3.1

运行时：**A100 高 RAM**（sm_80, 40GB VRAM / 83GB RAM）

## v3 相对 v2 的改动

| 类别 | 变化 |
| --- | --- |
| **修复 Cell 4 下载失败** | 原代码把 HF 缓存里的**符号链接本身**硬链过去，链接内容是相对路径 `../../../blobs/<sha>`，换目录后立刻悬空，于是 `getsize` 报 `[Errno 2] No such file or directory`。现在先 `realpath` 解析成真实 blob 再硬链，并自动清理上一轮留下的坏链接 |
| **A100 适配** | `nvfp4` 文本编码器只有 Blackwell(sm_120) 能跑，A100 自动改用 `qwen3vl_32b_minimax_h3_int8_convrot`（27.1GB）；`fp8_scaled` 需要 sm_89+，同样不选 |
| **A100 显存策略** | 删掉 `--highvram`（40GB 装不下 34GB DiT + 27GB TE）。ComfyUI 没有 `--normalvram` 这个参数，普通模式就是不传显存档位，现为 `--cache-none --disable-smart-memory` |
| **启动参数自检** | Cell 5 先跑 `main.py --help` 取出本次 master 真实支持的参数，`extra_args` 里不认识的会被剔除并提示，不再因一个拼错的 flag 直接启动失败；日志出现 `main.py: error:` 也会立即报出，不再白等 8 分钟 |
| **新增 Turbo 4 步加速 LoRA** | 自动下载 `larryvrh/MiniMax-H3-Turbo-Lora`，纯 Python 改键名前缀，落盘为 `minimax_h3_turbo_4步加速_comfyui.safetensors` |
| **新增双时钟采样器** | `shuaixn/ComfyUI-MiniMaxH3DualClockSampler`，修复 4 步下音频爆音，并提供旁路 LoRA 加载器 |
| **删除旧 Cell 5** | 图片/音频/视频一律在 ComfyUI 界面里上传；工作流模板由 ComfyUI 自带的 `comfyui-workflow-templates` 离线提供 |
| `TORCH_CUDA_ARCH_LIST` | 编译 SageAttention 2 时按实测 capability 传，不再写死 `12.0` |

## ⚠️ Turbo LoRA 的硬性前提

Turbo LoRA 补的是**完整 AdaLN 投影** `blocks.*.adaln_proj.linear.weight (96768, 2688)`。
所有 `pruned` 权重把它压成 `(96768, 8)`，会在 50 个 transformer block + final layer 共 **51 处报形状不匹配**，LoRA 只生效一部分，音频受损。

所以 `task = "fl2va_turbo"` 会自动改下 **非剪枝** 的 `minimax_h3_fl2va_int8_convrot.safetensors`（34GB）。
想跑官方 R2V 参考图模板就把 `task` 改成 `"ref2va"`，那条路不挂 LoRA。

## 执行顺序

**Cell 1 → 5**。重启 ComfyUI 界面时只重跑 **Cell 1 + Cell 5**。

In [ ]:
# ==========================================================
# Cell 1: 全局配置 + 工具函数（每次连接运行时都要先跑这一格）
# 目标运行时：Colab A100 40GB / 高 RAM
# ==========================================================
import json, os, shutil, subprocess, sys, time

CFG = {
    # --- 任务模式 ---
    #   "fl2va_turbo" : 图生视频 / 首尾帧 + Turbo 4 步加速 LoRA
    #                   -> 强制使用非剪枝 minimax_h3_fl2va_int8_convrot (34GB)
    #   "ref2va"      : 官方参考图生视频 R2V 模板 (21GB 剪枝模型, 不挂 LoRA)
    "task": "fl2va_turbo",

    # --- 路径 ---
    "comfy_dir": "/content/ComfyUI",
    # HF 缓存放 /content，和 models/ 同一文件系统，硬链接才不会把磁盘占两份
    "hf_home": "/content/hf_cache",

    # --- 权重文件：留空 = 按 GPU 架构自动选，填了就以填的为准 ---
    #   仓库 Comfy-Org/MiniMax-H3 内的相对路径，例如
    #   "diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors"
    "dit_file": "",
    "te_file": "",

    # --- Turbo 4 步加速 LoRA ---
    "lora_repo": "larryvrh/MiniMax-H3-Turbo-Lora",
    "lora_src": "minimax_h3_turbo_4step.safetensors",   # 非 EMA 版；EMA 版偏软，暂不推荐
    "lora_out": "minimax_h3_turbo_4步加速_comfyui.safetensors",
    "lora_force_on_pruned": False,  # True = 明知 51 处形状不匹配也要挂上去

    # --- FRP 内网穿透 ---
    "frp_host": "usoren.usdream.dpdns.org",
    "frp_port": 7000,
    "frp_token": "",        # frps 有 auth.token 才填
    "remote_port": 8091,
    "local_port": 8188,
    "frp_ver": "0.56.0",

    # --- Sage Attention ---
    #   "triton": pip 装 sageattention 1.x（1~2 分钟，A100 可用，推荐）
    #   "v2":     源码编译 SageAttention 2.2（10~20 分钟，按实测 capability 编）
    #   "skip":   不装（工作流里 PathchSageAttentionKJ 节点必须 Ctrl+B Bypass）
    "sage_mode": "triton",

    # --- 启动参数（A100 40GB）---
    #   显存档位只有 --gpu-only / --highvram / --lowvram / --novram / --cpu，
    #   普通模式（normal vram）就是**一个都不传**，没有 --normalvram 这个参数。
    #   34GB DiT + 27GB 文本编码器塞不进 40GB 显存，靠默认的自动换进换出。
    #   想提速可去掉 --disable-smart-memory；显存告急再加 --lowvram。
    #   Cell 5 启动前会用 main.py --help 对一遍，不认识的参数会被剔掉并提示。
    "extra_args": "--cache-none --disable-smart-memory",
}
CFG["access_url"] = "http://%s:%d" % (CFG["frp_host"], CFG["remote_port"])
CFG["frp_dir"] = "/content/frp_%s_linux_amd64" % CFG["frp_ver"]

os.makedirs(CFG["hf_home"], exist_ok=True)
os.environ["HF_HOME"] = CFG["hf_home"]


def save_cfg():
    with open("/content/h3_cfg.json", "w") as f:
        json.dump(CFG, f, indent=2, ensure_ascii=False)


def log(msg, tag="*"):
    print("[%s] %s" % (tag, msg), flush=True)


def sh(cmd, cwd=None, check=True, quiet=False):
    """运行 shell 命令；失败抛出，不静默继续。"""
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.stdout and not quiet:
        print(r.stdout.strip()[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError("命令失败(%d): %s" % (r.returncode, cmd))
    return r.stdout or ""


def pip(pkgs):
    sh("%s -m pip install -q %s" % (sys.executable, pkgs), quiet=True)


def free_gb(path="/content"):
    return shutil.disk_usage(path).free / 1024 ** 3


save_cfg()
log("任务模式: %s" % CFG["task"])
log("配置已写入 /content/h3_cfg.json")
log("访问地址将是: " + CFG["access_url"])
log("当前可用磁盘: %.0f GB" % free_gb())

In [ ]:
# ==========================================================
# Cell 2: ComfyUI 本体 (master 分支) + GPU 自检
# MiniMaxH3ReferenceToVideo 来自 PR #15224，必须用 master 最新提交
# 注意：本格全程**不在 Colab 内核里 import torch**
#   1) reload(torch) 会重复注册 TORCH_LIBRARY("triton") 直接 RuntimeError
#   2) 内核里 import torch 会帮 CUDA context 占掉 1~2GB 显存
# 所以 GPU 信息一律放到子进程里探测。
# ==========================================================
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as pkg_version

COMFY = CFG["comfy_dir"]

if not os.path.exists(COMFY):
    log("clone ComfyUI ...")
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI " + COMFY)
else:
    log("更新 ComfyUI ...")
    sh("git fetch --depth 1 origin master && git checkout -q master "
       "&& git reset --hard -q origin/master", cwd=COMFY)

log("当前提交: " + sh("git log -1 --format='%h %ad %s' --date=short",
                       cwd=COMFY, quiet=True).strip())


def torch_on_disk():
    """读磁盘上已安装的 torch 版本，不导入模块。"""
    try:
        return pkg_version("torch")
    except PackageNotFoundError:
        return None


_before = torch_on_disk()
log("安装 requirements（安装后校验 torch 是否被改动）...")
# hf_xet: Comfy-Org 仓库已走 Xet 后端，日志里的 "reconstructing file" 就是它
# 不装 hf_transfer：两者同时开会打架，Xet 实测 300MB/s+ 已够
pip("-r %s/requirements.txt huggingface_hub hf_xet" % COMFY)
_after = torch_on_disk()

if _before and _after and _before != _after:
    log("torch 被 requirements 改动：%s -> %s" % (_before, _after), "!")
    log("若下面探测报 kernel image 不兼容，"
        "恢复 Colab 自带版本：pip install -q torch==%s" % _before, "!")

# ---------- 子进程探测 GPU ----------
PROBE = "\n".join([
    "import json, torch",
    "info = {'torch': torch.__version__, 'cuda': torch.version.cuda,",
    "        'avail': torch.cuda.is_available()}",
    "if info['avail']:",
    "    p = torch.cuda.get_device_properties(0)",
    "    info['name'] = torch.cuda.get_device_name(0)",
    "    info['cap'] = list(torch.cuda.get_device_capability(0))",
    "    info['vram'] = round(p.total_memory / 1024 ** 3)",
    "    info['archs'] = torch.cuda.get_arch_list()",
    "print('PROBE=' + json.dumps(info))",
])
with open("/content/probe_gpu.py", "w") as f:
    f.write(PROBE + "\n")

out = sh("%s /content/probe_gpu.py" % sys.executable, check=False, quiet=True)
probe = None
for line in out.splitlines():
    if line.startswith("PROBE="):
        probe = json.loads(line[6:])

if probe is None:
    log("torch 探测失败，子进程输出：\n" + out[-2000:], "!")
    raise RuntimeError("torch 不可用，请看上方输出")

log("torch: %s | cuda: %s" % (probe["torch"], probe["cuda"]))
if not probe["avail"]:
    raise RuntimeError("未检测到 GPU：请把运行时改成 A100 高 RAM")

cap = tuple(probe["cap"])
sm = cap[0] * 10 + cap[1]
CFG["cap"] = list(cap)
CFG["sm"] = sm
CFG["vram_gb"] = probe["vram"]
log("GPU: %s | capability: sm_%d | VRAM: %d GB" % (probe["name"], sm, probe["vram"]))

# ---------- 按架构告知量化格式可用性 ----------
#   nvfp4  需 sm_120+ (Blackwell)
#   fp8    需 sm_89+  (Ada/Hopper/Blackwell)
#   int8   sm_75+ 普遍可用  <- A100 (sm_80) 走这条
CFG["can_nvfp4"] = sm >= 120
CFG["can_fp8"] = sm >= 89
if sm == 80:
    log("A100 (sm_80) 已识别：nvfp4 / fp8 均无原生内核，Cell 4 会自动改用 int8_convrot")
elif not CFG["can_nvfp4"]:
    log("非 Blackwell：nvfp4 文本编码器不可用，Cell 4 会改用 int8_convrot", "!")
else:
    log("Blackwell：nvfp4 / int8 均可原生运行")

if not any("sm_%d" % sm in a for a in probe.get("archs", [])):
    log("当前 torch 编译目标不含 sm_%d：%s，高概率会报 kernel image 错误"
        % (sm, probe.get("archs")), "!")

# ---------- 核心节点存在性校验（grep 源码，不启动服务） ----------
hit = sh("grep -rl MiniMaxH3ReferenceToVideo %s/comfy_extras %s/nodes.py || true"
         % (COMFY, COMFY), check=False, quiet=True).strip()
log("MiniMaxH3ReferenceToVideo 已就绪" if hit
    else "未找到 MiniMaxH3ReferenceToVideo：代码不够新，请重跑本格",
    "*" if hit else "!")

save_cfg()
log("Cell 2 完成")

In [ ]:
# ==========================================================
# Cell 3: 自定义节点 + Sage Attention
#   PathchSageAttentionKJ -> kijai/ComfyUI-KJNodes
#   VHS_LoadVideo         -> Kosinkadink/ComfyUI-VideoHelperSuite
#   双时钟采样器           -> shuaixn/ComfyUI-MiniMaxH3DualClockSampler
#        修复 Turbo LoRA 在 4 步下的音频爆音/削波（视频 shift=12 / 音频 shift=3
#        必须分开积分），同时提供 Load LoRA (Bypass, Model Only) 旁路加载器
#   ComfyUI-Manager       -> 可选，排查缺失节点用
# 同样不在内核里 import sageattention/torch，一律用子进程验证。
# ==========================================================
NODES = [
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git",
    "https://github.com/ltdrdata/ComfyUI-Manager.git",
]
if CFG["task"] == "fl2va_turbo":
    NODES.insert(0, "https://github.com/shuaixn/ComfyUI-MiniMaxH3DualClockSampler.git")

cn_dir = os.path.join(COMFY, "custom_nodes")
os.makedirs(cn_dir, exist_ok=True)

for repo in NODES:
    name = repo.rstrip("/").split("/")[-1].replace(".git", "")
    path = os.path.join(cn_dir, name)
    if os.path.exists(path):
        log("更新 " + name)
        sh("git pull -q", cwd=path, check=False, quiet=True)
    else:
        log("安装 " + name)
        sh("git clone --depth 1 -q %s %s" % (repo, path))
    req = os.path.join(path, "requirements.txt")
    if os.path.exists(req):
        # 过滤掉会连带升级/降级 torch 与 triton 的行（保住 Colab 原版）
        skip = ("torch", "torchvision", "torchaudio", "triton", "#")
        keep = [l for l in open(req)
                if l.strip() and not l.lower().lstrip().startswith(skip)]
        if keep:
            with open("/tmp/req.txt", "w") as f:
                f.writelines(keep)
            pip("-r /tmp/req.txt")


def sage_available():
    """在子进程里真实 import 一次，返回 (是否可用, 详情)。"""
    code = ("import sageattention as s;"
            "print('SAGE_OK=' + getattr(s, '__version__', '1.x'))")
    out = sh('%s -c "%s"' % (sys.executable, code), check=False, quiet=True)
    for line in out.splitlines():
        if line.startswith("SAGE_OK="):
            return True, line[8:].strip()
    return False, out.strip()[-500:]


mode = CFG["sage_mode"]
if mode == "skip":
    CFG["sage_ok"] = False
    log("按配置跳过 sageattention；请在界面里把 Patch Sage Attention KJ "
        "节点设为 Bypass (Ctrl+B)，否则仍会报错", "!")
else:
    ok, detail = sage_available()
    if ok:
        log("sageattention 已安装，跳过：" + detail)
    elif mode == "v2":
        arch = "%d.%d" % tuple(CFG["cap"])          # A100 -> 8.0，不再写死 12.0
        log("源码编译 SageAttention 2.2（arch %s，约 10~20 分钟）..." % arch)
        src = "/content/SageAttention"
        if not os.path.exists(src):
            sh("git clone --depth 1 -q https://github.com/thu-ml/SageAttention " + src)
        env = "TORCH_CUDA_ARCH_LIST=%s MAX_JOBS=8 EXT_PARALLEL=4" % arch
        sh("%s %s -m pip install -q . --no-build-isolation" % (env, sys.executable),
           cwd=src, check=False)
    else:
        log("安装 sageattention 1.x（纯 Triton 实现，无需编译，不动 torch）...")
        pip("--no-deps sageattention==1.0.6")

    if not ok:
        ok, detail = sage_available()
        if ok:
            log("sageattention 可用：" + detail)
        else:
            log("sageattention 仍不可用，子进程输出：\n" + detail, "!")
            log("两条路：把 Cell 1 的 sage_mode 改成 'v2' 重装，"
                "或把该节点 Ctrl+B Bypass", "!")
    CFG["sage_ok"] = ok

if not CFG.get("can_fp8", False):
    log("A100 上 Patch Sage Attention KJ 的 sage_attention 选 'auto' 即可，"
        "不要选 fp8 三角内核（sm_89+ 才有）；allow_compile 保持 False", "!")

save_cfg()
log("Cell 3 完成（新节点需重启 ComfyUI 才会加载，即重跑 Cell 5）")

In [ ]:
# ==========================================================
# Cell 4: 模型下载 + Turbo LoRA（并行 + 断点续传 + 自动选精度）
#
# 修复上一版的 [Errno 2] No such file or directory：
#   hf_hub_download 返回的是 HF 缓存里的**符号链接**
#     snapshots/<rev>/vae/x.safetensors -> ../../../blobs/<sha>
#   Linux 下 os.link() 不跟随符号链接，硬链过去的是链接本身，
#   而它的目标是相对路径，换到 models/vae/ 下就指向了不存在的 /content/blobs/...
#   → 目录里看得到 lrwxrwxrwx，但 getsize() 直接 ENOENT。
#   修法：先 os.path.realpath() 解成真实 blob 再硬链，并清掉旧的悬空链接。
# ==========================================================
import struct
from concurrent.futures import ThreadPoolExecutor, as_completed

os.environ["HF_HOME"] = CFG["hf_home"]
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # Xet 后端已够快，两者同开会打架
os.makedirs(CFG["hf_home"], exist_ok=True)

from huggingface_hub import hf_hub_download

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        log("已读取 HF_TOKEN")
except Exception:
    log("未读到 HF_TOKEN（左侧 Secrets）：公开仓库仍可下载", "!")

REPO = "Comfy-Org/MiniMax-H3"
sm = CFG.get("sm", 80)
turbo = CFG["task"] == "fl2va_turbo"

# ---------- 1. 按 GPU 架构选权重 ----------
# DiT：Turbo LoRA 补的是完整 AdaLN 投影 (96768, 2688)，pruned 版只有 (96768, 8)，
#      51 处形状不匹配 → 想用 LoRA 就必须下非剪枝版
if turbo:
    dit = "diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors"        # 34 GB
else:
    dit = "diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors"  # 21 GB
# 文本编码器：nvfp4 需 sm_120+，A100 只能 int8_convrot
te = ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" if sm >= 120
      else "text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors")   # 15.7 / 27.1 GB

dit = CFG["dit_file"] or dit
te = CFG["te_file"] or te
CFG["dit_file"], CFG["te_file"] = dit, te

TASKS = [
    (REPO, dit, "diffusion_models", None),
    (REPO, te, "text_encoders", None),
    (REPO, "vae/minimax_h3_video_vae_fp16.safetensors", "vae", None),
    (REPO, "vae/minimax_h3_audio_vae_fp32.safetensors", "vae", None),
]
if turbo:
    TASKS.append((CFG["lora_repo"], CFG["lora_src"], "loras", CFG["lora_out"]))

log("DiT          : " + os.path.basename(dit))
log("文本编码器   : " + os.path.basename(te))
if turbo:
    log("Turbo LoRA   : %s  ->  %s" % (CFG["lora_src"], CFG["lora_out"]))
    if "pruned" in dit and not CFG["lora_force_on_pruned"]:
        raise RuntimeError(
            "dit_file 指向了 pruned 权重，Turbo LoRA 会在 51 处 adaln_proj 报形状不匹配。"
            "换成非剪枝版，或把 lora_force_on_pruned 设为 True 强行继续。")

EST = 34 + 27 + 6 + 1 if (turbo and sm < 120) else 21 + 16 + 6
log("预估占盘 ≈ %d GB（HF 缓存与 models/ 同盘，硬链接不翻倍），当前可用 %.0f GB"
    % (EST, free_gb()))
if free_gb() < EST + 8:
    log("磁盘不够，下载到一半可能断。考虑改用 pruned 权重或清理 /content", "!")


# ---------- 2. 链接工具 ----------
def link_real(src, final):
    """把 HF 缓存里的文件落到 models/ 下，不复制一份。
    关键：先 realpath 解成真实 blob，否则硬链到的是相对符号链接。"""
    real = os.path.realpath(src)
    if not os.path.isfile(real):
        raise RuntimeError("HF 缓存解析失败: %s -> %s" % (src, real))
    # 清掉上一轮留下的悬空链接（lexists=True 而 exists=False）
    if os.path.lexists(final) and not os.path.exists(final):
        os.unlink(final)
    if os.path.lexists(final):
        return real
    try:
        os.link(real, final)            # 同盘硬链，0 额外占用
    except OSError:
        try:
            os.symlink(real, final)     # 跨盘：绝对路径符号链接
        except OSError:
            shutil.copy2(real, final)
    return real


def safetensors_ok(path):
    """只读 8 字节头长 + JSON 头，确认文件真的能读且完整。"""
    size = os.path.getsize(os.path.realpath(path))
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        if n <= 0 or n + 8 > size:
            return False, size
        json.loads(f.read(n))
    return True, size


def fetch(repo, filename, subdir, rename):
    dest_dir = os.path.join(COMFY, "models", subdir)
    os.makedirs(dest_dir, exist_ok=True)
    base = rename or os.path.basename(filename)
    final = os.path.join(dest_dir, base)

    if os.path.exists(final) and os.path.getsize(os.path.realpath(final)) > 100 * 1024 ** 2:
        return "跳过(已存在) %s  %.1f GB" % (
            base, os.path.getsize(os.path.realpath(final)) / 1024 ** 3)

    src = hf_hub_download(repo_id=repo, filename=filename, repo_type="model")

    if rename and repo == CFG["lora_repo"]:
        # LoRA 需要改键名前缀，不能直接链
        add_diffusion_prefix(os.path.realpath(src), final)
    else:
        link_real(src, final)

    ok, size = safetensors_ok(final)
    if not ok:
        raise RuntimeError("%s 头部校验失败，文件可能不完整" % base)
    return "完成 %s  %.1f GB" % (base, size / 1024 ** 3)


# ---------- 3. Turbo LoRA 键名转换（纯 Python，不 import torch） ----------
def add_diffusion_prefix(src, dst, prefix="diffusion_model."):
    """作者的 LoRA 键是 blocks.0.attn.qkv_proj.lora_A.weight，
    ComfyUI 的补丁系统需要 diffusion_model. 命名空间，否则满屏 lora key not loaded。
    safetensors = 8 字节头长 + JSON 头 + 数据区，偏移量相对数据区起点，
    所以只重写 JSON 头、数据区原样拷贝即可，dtype/shape/rank/强度全不变。"""
    tmp = dst + ".part"
    with open(src, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        head = json.loads(f.read(n))
        meta = head.pop("__metadata__", None)
        renamed = 0
        new = {}
        for k, v in head.items():
            if k.startswith(prefix):
                new[k] = v
            else:
                new[prefix + k] = v
                renamed += 1
        if meta is not None:
            new = dict([("__metadata__", meta)] + list(new.items()))
        hb = json.dumps(new, separators=(",", ":")).encode("utf-8")
        hb += b" " * ((-len(hb)) % 8)
        with open(tmp, "wb") as o:
            o.write(struct.pack("<Q", len(hb)))
            o.write(hb)
            shutil.copyfileobj(f, o, 8 * 1024 ** 2)
    os.replace(tmp, dst)
    log("LoRA 键名已加前缀：%d/%d 个 tensor" % (renamed, len(head)))


# ---------- 4. 开跑 ----------
t0 = time.time()
failed = []
with ThreadPoolExecutor(max_workers=2) as ex:
    futs = {ex.submit(fetch, r, f, d, rn): f for r, f, d, rn in TASKS}
    for fu in as_completed(futs):
        try:
            log(fu.result())
        except Exception as e:
            failed.append(futs[fu])
            log("下载失败 %s -> %r" % (futs[fu], e), "!")

# ---------- 5. 核对清单（跟随符号链接看真实大小） ----------
log("模型清单（-L 跟随链接）：")
sh("ls -lLh %s/models/diffusion_models %s/models/text_encoders %s/models/vae %s/models/loras"
   % (COMFY, COMFY, COMFY, COMFY), check=False)

bad = []
for _r, f, d, rn in TASKS:
    p = os.path.join(COMFY, "models", d, rn or os.path.basename(f))
    if not os.path.exists(p):
        bad.append(p)
if bad or failed:
    log("以下文件仍不可用，重跑本格即可断点续传：\n  " + "\n  ".join(bad + failed), "!")
else:
    log("全部文件就绪")

save_cfg()
log("耗时 %.1f 分钟，剩余磁盘 %.0f GB" % ((time.time() - t0) / 60, free_gb()))

In [ ]:
# ==========================================================
# Cell 5: frpc.toml + start ComfyUI (restart UI = rerun Cell 1 + this cell)
# All uploads happen inside the ComfyUI web UI. No colab uploads here.
# ==========================================================
import configparser, re, shlex, threading

CFG = json.load(open("/content/h3_cfg.json"))
COMFY, FRP_DIR = CFG["comfy_dir"], CFG["frp_dir"]
assert os.path.isdir(COMFY), "missing %s, run Cell 2 first" % COMFY

# ---------- 1. frp binary ----------
if not os.path.exists(FRP_DIR + "/frpc"):
    log("download frp %s ..." % CFG["frp_ver"])
    _v = CFG["frp_ver"]
    sh("wget -qO- https://github.com/fatedier/frp/releases/download/v" + _v +
       "/frp_" + _v + "_linux_amd64.tar.gz | tar -xz -C /content")
assert os.path.exists(FRP_DIR + "/frpc"), "frpc download failed, rerun this cell"
sh("chmod +x %s/frpc" % FRP_DIR, quiet=True)

# ---------- 2. frpc.toml ----------
conf = [
    'serverAddr = "%s"' % CFG["frp_host"],
    "serverPort = %d" % CFG["frp_port"],
    "loginFailExit = false",
    "transport.tcpMux = true",
    "transport.poolCount = 5",
    'log.to = "/content/frpc.log"',
    'log.level = "info"',
]
if CFG.get("frp_token"):
    conf.insert(2, 'auth.token = "%s"' % CFG["frp_token"])
conf += [
    "",
    "[[proxies]]",
    'name = "comfyui_colab"',
    'type = "tcp"',
    'localIP = "127.0.0.1"',
    "localPort = %d" % CFG["local_port"],
    "remotePort = %d" % CFG["remote_port"],
]
open(FRP_DIR + "/frpc.toml", "w").write("\n".join(conf) + "\n")
log("frpc.toml written")

# ---------- 3. keep-alive ----------
def _keep():
    while True:
        time.sleep(300)
        print("[keep-alive]", flush=True)

threading.Thread(target=_keep, daemon=True).start()

# ---------- 4. start frpc ----------
for f in ("/content/comfy.log", "/content/frpc.log"):
    if os.path.exists(f):
        os.remove(f)
sh("pkill -f frpc || true; pkill -f 'ComfyUI/main.py' || true", check=False, quiet=True)
subprocess.Popen("%s/frpc -c %s/frpc.toml >> /content/frpc.log 2>&1"
                 % (FRP_DIR, FRP_DIR), shell=True)
time.sleep(6)
frp_log = ""
if os.path.exists("/content/frpc.log"):
    frp_log = open("/content/frpc.log", errors="ignore").read()
HINTS = [
    ("start proxy success", "FRP tunnel up -> " + CFG["access_url"]),
    ("token in login", "frps requires a token: fill frp_token in Cell 1"),
    ("already used", "remote port in use: change remote_port in Cell 1"),
    ("port not allowed", "port outside frps allowPorts: change port or server config"),
]
for key, msg in HINTS:
    if key in frp_log:
        log(msg, "*" if key == "start proxy success" else "!")
        break
else:
    log("no 'start proxy success' yet, tail of log:\n" + frp_log[-800:], "!")

# ---------- 5. Manager offline mode ----------
for p in (COMFY + "/user/__manager/config.ini",
          COMFY + "/user/default/ComfyUI-Manager/config.ini",
          COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"):
    os.makedirs(os.path.dirname(p), exist_ok=True)
    cp = configparser.ConfigParser()
    if os.path.exists(p):
        cp.read(p)
    cp.setdefault("default", {})
    cp["default"]["network_mode"] = "private"
    cp.write(open(p, "w"))
log("ComfyUI-Manager set to offline mode")

# ---------- 6. validate launch flags against this ComfyUI build ----------
# ComfyUI has no --normalvram: normal mode is simply passing no vram flag at all.
# Flags also come and go between master commits, so ask main.py --help what it
# actually accepts and drop anything unknown instead of dying on argparse.
def supported_flags():
    out = sh("%s main.py --help" % sys.executable, cwd=COMFY, check=False, quiet=True)
    return set(re.findall(r"--[A-Za-z0-9][A-Za-z0-9_.-]*", out))


def sanitize(extra, known):
    if not known:
        return extra, []
    toks, keep, dropped, i = shlex.split(extra), [], [], 0
    while i < len(toks):
        t = toks[i]
        if t.startswith("--"):
            name = t.split("=", 1)[0]
            good = name in known
            (keep if good else dropped).append(t)
            i += 1
            while i < len(toks) and not toks[i].startswith("-"):
                if good:
                    keep.append(toks[i])
                i += 1
        else:
            keep.append(t)
            i += 1
    return " ".join(keep), dropped


known = supported_flags()
extra, dropped = sanitize(CFG["extra_args"], known)
if dropped:
    log("this ComfyUI build does not accept %s -> removed. Fix Cell 1 extra_args."
        % " ".join(dropped), "!")
if "--lowvram" not in extra and "--highvram" not in extra:
    log("vram mode: normal (default, no flag)")

# ---------- 7. launch ComfyUI ----------
# No --use-sage-attention: the Patch Sage Attention KJ node inside the workflow
# already switches attention, adding both would patch twice.
launch = ("python main.py --listen 127.0.0.1 --port %d --enable-cors-header '*' "
          "--preview-method auto %s" % (CFG["local_port"], extra))
log("launch: " + launch)
subprocess.Popen(launch + " > /content/comfy.log 2>&1", shell=True, cwd=COMFY)

ready = False
for i in range(240):
    time.sleep(2)
    txt = ""
    if os.path.exists("/content/comfy.log"):
        txt = open("/content/comfy.log", errors="ignore").read()
    if "To see the GUI go to" in txt:
        ready = True
        break
    if ("Traceback" in txt and i > 10) or "main.py: error:" in txt:
        log("startup error, tail of log:\n" + txt[-3000:], "!")
        break

BAR = "=" * 62
print("\n" + BAR)
print("ComfyUI %s : %s" % ("READY" if ready else "NOT READY", CFG["access_url"]))
print("URL must include :%d  (without the port you get the frps 404 page)"
      % CFG["remote_port"])
print("Upload image/audio/video : use the upload button on LoadImage / VHS_LoadVideo")
print("Official templates       : Workflow -> Browse Templates -> Video -> MiniMax H3")
print("Text encoder to select   : " + os.path.basename(CFG["te_file"]))
print("   (templates default to nvfp4; on A100 switch CLIPLoader to the file above)")
if CFG["task"] == "fl2va_turbo":
    print("Turbo LoRA               : %s   strength 1.0" % CFG["lora_out"])
    print("   sampler MiniMax H3 Dual-Clock Euler / scheduler simple / steps 4 or 8")
    print("   denoise 1.0 / BasicGuider / video shift 12 / audio shift 3")
print(BAR + "\n")
if not ready:
    print(open("/content/comfy.log", errors="ignore").read()[-3000:])

subprocess.run("tail -f /content/comfy.log", shell=True)

## 报错定位速查

| 报错 | 原因 | 处理 |
| --- | --- | --- |
| `[Errno 2] No such file or directory: .../models/vae/xxx.safetensors`，但 `ls` 里看得到 `lrwxrwxrwx` | 把 HF 缓存的**相对符号链接**硬链过去了，换目录后悬空 | v3 已修（`os.path.realpath` 后再链）。手动清：`find /content/ComfyUI/models -xtype l -delete` 然后重跑 Cell 4 |
| `main.py: error: unrecognized arguments: --xxx` | `extra_args` 里有本版 ComfyUI 不认识的参数。显存档位**只有** `--gpu-only / --highvram / --lowvram / --novram / --cpu`，没有 `--normalvram` | v3.1 已剔除并加了启动前自检。自己改参数前先 `cd /content/ComfyUI && python main.py --help` 看一眼 |
| `lora key not loaded: blocks.0.attn...` 刷屏 | LoRA 键名没加 `diffusion_model.` 前缀，或文件放进了 `diffusion_models/` | 重跑 Cell 4；确认文件在 `models/loras/` |
| `adaln_proj.linear.weight shape '[96768, 8]' is invalid for input of size ...` | 主模型是 **pruned** 版，Turbo LoRA 需要完整 `(96768, 2688)` | Cell 1 的 `task` 保持 `"fl2va_turbo"`（会自动下非剪枝 34GB），或改 `"ref2va"` 不挂 LoRA |
| `requires a two-stream NestedTensor ... latent has type Tensor` | 双时钟节点版本旧，入口类型错 | `cd custom_nodes/ComfyUI-MiniMaxH3DualClockSampler && git pull` 后重跑 Cell 5 |
| `requires a full denoise schedule starting at sigma 1.0` | `denoise` < 1 | 把 denoise 改回 `1.0`，要弱化去调 LoRA strength |
| `only supports the validated MiniMax H3 shifts` | 别的节点通过 `transformer_options` 改了 shift | 视频 shift 回 12，音频 shift 回 3 |
| `CUDA error: no kernel image is available` / `sm_80 is not compatible` | ComfyUI requirements 把 torch 改掉了 | 按 Cell 2 打印的旧版本号 `pip install -q torch==<旧版本>`，重启运行时 |
| `Only a single TORCH_LIBRARY can be used to register the namespace triton` | 在内核里 reload 了 torch | 不要在 Cell 里 `import torch`；已发生就重启运行时，只跑 Cell 1 + 5 |
| CUDA OOM（采样阶段） | 40GB 显存装不下 34GB DiT + 中间激活 | 降分辨率到 960×544；缩短时长到 4~5s；确认 `--disable-smart-memory` 在 |
| 系统 RAM 耗尽、会话崩溃 | 34GB DiT + 27GB 文本编码器同时驻 RAM，83GB 很紧 | 保持 `--cache-none`；先跑一次只提示编码、再接采样；实在不行把 `task` 改回 `"ref2va"` |
| 模型下拉框里没有刚下的文件 | ComfyUI 启动后才扫目录 | 界面右上角 Refresh，或重跑 Cell 5 |
| 打开网址是 404 | 地址没带端口 | 必须是 `http://<host>:8091` |

---

## A100 40GB 推荐参数

| 项 | 值 | 说明 |
| --- | --- | --- |
| 分辨率 | **1344×768** | 官方验证档位。不建议 1920×1088，40GB 显存会 OOM |
| 时长 | 5～8 s | `PrimitiveFloat` 节点；帧数自动对齐到 17 的倍数 |
| fps | 24 | `CreateVideo` |
| steps | **4**（Turbo）/ 8（更干净）/ 20～25（不挂 LoRA） | |
| scheduler | `simple` | 不要用 Beta |
| sampler | `MiniMax H3 Dual-Clock Euler` | 4 步下不要用 `res_multistep`，会出彩色闪光 |
| denoise | `1.0` | 必须满去噪 |
| guider | `BasicGuider` | 等价 CFG=1.0 |
| video shift / audio shift | 12 / 3 | 已验证组合，不要改 |
| LoRA strength | 1.0 | |

4 步的 sigma 网格：`[1.0000, 0.9730, 0.9231, 0.8000, 0.0000]`。
成功日志长这样：

```
[MiniMaxH3DualClock] Euler sampling with video shift 12.0, audio shift 3.0, steps 4
```

---

## 工作流里要手动改的两个控件

1. **CLIPLoader** → 官方模板默认填 `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`，
   A100 上改成 `qwen3vl_32b_minimax_h3_int8_convrot.safetensors`（`type` 保持 `minimax`）。
2. **UNETLoader** → 改成 Cell 4 实际下的那个 `.safetensors`（启动日志里会打印文件名）。

LoRA 加载器推荐用 `Load LoRA (Bypass, Model Only)`（双时钟节点包里自带），
方便 Ctrl+B 一键对比开/关 LoRA 的效果。